# EGNN Generative Model: INACTIVE → ACTIVE Transformation

**Objetivo:** entrenar una Equivariant Graph Neural Network (EGNN) que aprenda la transformación estructural INACTIVE → ACTIVE de conformaciones de kinasas, evaluada con esquema LOKO (Leave-One-Kinase-Out).

**Pipeline:**
1. Cargar e inspeccionar el dataset
2. Construir pares INACTIVE → ACTIVE por kinasa
3. Dataset class + collate function (con alineación correcta INACTIVE/ACTIVE)
4. Modelo EGNN
5. Funciones de loss, métricas, entrenamiento y validación
6. Carga de folds LOKO
7. Loop de entrenamiento y evaluación LOKO
8. Resumen final


In [2]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Seeds para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hiperparámetros
EPOCHS = 50
BATCH_SIZE = 1  # fijo en 1: el EGNNLayer sparse procesa un grafo por vez (ver Sección 5)
LEARNING_RATE = 1e-4
COORD_THRESHOLD = 2.0  # Å, umbral para structural recovery

print(f"✓ Device: {DEVICE}")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Epochs: {EPOCHS} | Batch size: {BATCH_SIZE} | LR: {LEARNING_RATE} | Coord threshold: {COORD_THRESHOLD} Å")


✓ Device: cpu
✓ PyTorch version: 2.12.1+cu130
✓ Epochs: 50 | Batch size: 1 | LR: 0.0001 | Coord threshold: 2.0 Å


## 1. Cargar e inspeccionar el dataset

In [3]:
print("Loading dataset_ready.pkl...")
with open('dataset_ready.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print(f"Raw data type: {type(raw_data)}")

def flatten_dataset(data):
    """Aplana el dataset a una lista de dicts, manejando estructuras anidadas."""
    result = []
    if isinstance(data, dict):
        for v in data.values():
            if isinstance(v, list):
                for item in v:
                    if isinstance(item, dict):
                        result.append(item)
                    elif isinstance(item, list) and len(item) > 0 and isinstance(item[0], dict):
                        result.extend(item)
            elif isinstance(v, dict):
                result.append(v)
    elif isinstance(data, list):
        for item in data:
            if isinstance(item, dict):
                result.append(item)
            elif isinstance(item, list) and len(item) > 0:
                if isinstance(item[0], dict):
                    result.extend(item)
                else:
                    result.append(item)
    return result

dataset = flatten_dataset(raw_data)
dataset = [s for s in dataset if isinstance(s, dict)]
print(f"✓ Dataset aplanado: {len(dataset)} muestras (dicts)")

if len(dataset) > 0:
    first = dataset[0]
    print(f"\nClaves de la primera muestra: {list(first.keys())}")
    for k, v in first.items():
        if isinstance(v, np.ndarray):
            print(f"  {k}: ndarray shape={v.shape}, dtype={v.dtype}")
        elif isinstance(v, (list, tuple)):
            try:
                arr = np.array(v)
                print(f"  {k}: secuencia shape={arr.shape}")
            except Exception:
                print(f"  {k}: secuencia (no numérica)")
        else:
            print(f"  {k}: {type(v).__name__} = {v}")


Loading dataset_ready.pkl...
Raw data type: <class 'dict'>
✓ Dataset aplanado: 797 muestras (dicts)

Claves de la primera muestra: ['sample_id', 'pdb_id', 'kinase_name', 'num_residues', 'ca_coords', 'adjacency', 'num_edges', 'conformation_class', 'sequence', 'embedding']
  sample_id: str = EGFR_3w33_A
  pdb_id: str = 3w33
  kinase_name: str = EGFR
  num_residues: int = 297
  ca_coords: ndarray shape=(297, 3), dtype=float32
  adjacency: ndarray shape=(297, 297), dtype=float32
  num_edges: float64 = 2568.0
  conformation_class: str = INACTIVE
  sequence: str = QALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELRKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDDDVVDADEYLIXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
  embedd

## 2. Estadísticas del dataset

In [4]:
kinase_confs = defaultdict(lambda: defaultdict(int))
coord_lengths = []
embed_dims = []

for sample in dataset:
    kinase = sample.get('kinase_name')
    conformation = sample.get('conformation_class')
    if kinase and conformation:
        kinase_confs[kinase][conformation] += 1
    if 'ca_coords' in sample:
        coord_lengths.append(len(np.array(sample['ca_coords'])))
    if 'embedding' in sample:
        emb = np.array(sample['embedding'])
        if emb.ndim == 2:
            embed_dims.append(emb.shape[1])
        elif emb.ndim == 3:
            embed_dims.append(emb.shape[2])

print("=" * 80)
print("DATASET STATISTICS")
print("=" * 80)
print(f"\nTotal samples: {len(dataset)}")
print(f"Unique kinases: {len(kinase_confs)}")

print("\nDistribución por kinasa:")
kinase_df = []
for kinase in sorted(kinase_confs.keys()):
    active = kinase_confs[kinase].get('ACTIVE', 0)
    inactive = kinase_confs[kinase].get('INACTIVE', 0)
    print(f"  {kinase:12s}: ACTIVE={active:3d}, INACTIVE={inactive:3d}, Total={active+inactive:3d}")
    kinase_df.append({'Kinase': kinase, 'ACTIVE': active, 'INACTIVE': inactive, 'Total': active+inactive})
kinase_df = pd.DataFrame(kinase_df)

print(f"\nTotal ACTIVE: {kinase_df['ACTIVE'].sum()}")
print(f"Total INACTIVE: {kinase_df['INACTIVE'].sum()}")

if coord_lengths:
    print(f"\nLongitud de coordenadas (n° de residuos):")
    print(f"  Min: {min(coord_lengths)}, Max: {max(coord_lengths)}, Media: {np.mean(coord_lengths):.1f}, Mediana: {np.median(coord_lengths):.1f}")
if embed_dims:
    print(f"\nDimensión de embedding: {set(embed_dims)}")


DATASET STATISTICS

Total samples: 797
Unique kinases: 6

Distribución por kinasa:
  ABL1        : ACTIVE= 13, INACTIVE= 35, Total= 48
  BRAF        : ACTIVE= 45, INACTIVE=148, Total=193
  EGFR        : ACTIVE=176, INACTIVE=196, Total=372
  FGFR1       : ACTIVE=121, INACTIVE=  4, Total=125
  KIT         : ACTIVE= 11, INACTIVE= 33, Total= 44
  PDGFRA      : ACTIVE=  2, INACTIVE=  8, Total= 10

Total ACTIVE: 368
Total INACTIVE: 424

Longitud de coordenadas (n° de residuos):
  Min: 234, Max: 614, Media: 288.0, Mediana: 287.0

Dimensión de embedding: {1280}


## 3. Construir pares INACTIVE → ACTIVE por kinasa

In [5]:
print("Building kinase lookup structures...")
kinase_samples = defaultdict(list)
kinase_conformations = defaultdict(lambda: defaultdict(list))

for sample in dataset:
    kinase = sample.get('kinase_name')
    conformation = sample.get('conformation_class')
    if kinase is None or conformation is None:
        continue
    kinase_samples[kinase].append(sample)
    kinase_conformations[kinase][conformation].append(sample)

print(f"✓ Unique kinases: {len(kinase_samples)}")
for kinase in sorted(kinase_samples.keys()):
    active_count = len(kinase_conformations[kinase].get('ACTIVE', []))
    inactive_count = len(kinase_conformations[kinase].get('INACTIVE', []))
    print(f"  {kinase}: {active_count} ACTIVE, {inactive_count} INACTIVE")


def create_pairs_for_kinase(kinase_name, max_length_diff=30, min_overlap_residues=50):
    """
    Crea pares ACTIVE/INACTIVE para una kinasa, priorizando:
    1. Misma kinasa
    2. Longitud de secuencia similar
    3. Máximo overlap de residuos
    """
    active_samples = kinase_conformations[kinase_name].get('ACTIVE', [])
    inactive_samples = kinase_conformations[kinase_name].get('INACTIVE', [])
    if not active_samples or not inactive_samples:
        return []

    pairs = []
    for active in active_samples:
        active_coords = np.array(active['ca_coords'])
        active_len = len(active_coords)

        best_inactive, best_score = None, -np.inf
        for inactive in inactive_samples:
            inactive_coords = np.array(inactive['ca_coords'])
            inactive_len = len(inactive_coords)

            length_diff = abs(active_len - inactive_len)
            if length_diff > max_length_diff:
                continue
            length_score = 1.0 / (1.0 + length_diff / 10.0)

            min_len = min(active_len, inactive_len)
            overlap_score = (min_len / max(active_len, inactive_len)) if min_len >= min_overlap_residues else 0

            score = 0.6 * length_score + 0.4 * overlap_score
            if score > best_score:
                best_score, best_inactive = score, inactive

        if best_inactive is not None and best_score > 0.3:
            pairs.append({'inactive': best_inactive, 'active': active, 'kinase': kinase_name, 'score': best_score})
    return pairs


print("\nCreando pares ACTIVE/INACTIVE por kinasa...")
all_pairs = {}
for kinase in sorted(kinase_samples.keys()):
    pairs = create_pairs_for_kinase(kinase)
    all_pairs[kinase] = pairs
    print(f"  {kinase}: {len(pairs)} pares")

total_pairs = sum(len(p) for p in all_pairs.values())
print(f"\n✓ Total de pares creados: {total_pairs}")


Building kinase lookup structures...
✓ Unique kinases: 6
  ABL1: 13 ACTIVE, 35 INACTIVE
  BRAF: 45 ACTIVE, 148 INACTIVE
  EGFR: 176 ACTIVE, 196 INACTIVE
  FGFR1: 121 ACTIVE, 4 INACTIVE
  KIT: 11 ACTIVE, 33 INACTIVE
  PDGFRA: 2 ACTIVE, 8 INACTIVE

Creando pares ACTIVE/INACTIVE por kinasa...
  ABL1: 13 pares
  BRAF: 45 pares
  EGFR: 176 pares
  FGFR1: 121 pares
  KIT: 11 pares
  PDGFRA: 2 pares

✓ Total de pares creados: 368


### Tabla resumen de pares por kinasa

In [6]:
print("\n" + "=" * 70)
print("PAIR SUMMARY BY KINASE")
print("=" * 70)

summary_data = []
for kinase in sorted(all_pairs.keys()):
    active_count = len(kinase_conformations[kinase].get('ACTIVE', []))
    inactive_count = len(kinase_conformations[kinase].get('INACTIVE', []))
    summary_data.append({
        'Kinase': kinase, 'Active': active_count,
        'Inactive': inactive_count, 'Pairs': len(all_pairs[kinase])
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print(f"\n{'='*70}")
print(f"Total structures available: {len(dataset)}")
print(f"Total pairs available: {total_pairs}")
print(f"{'='*70}\n")



PAIR SUMMARY BY KINASE
Kinase  Active  Inactive  Pairs
  ABL1      13        35     13
  BRAF      45       148     45
  EGFR     176       196    176
 FGFR1     121         4    121
   KIT      11        33     11
PDGFRA       2         8      2

Total structures available: 797
Total pairs available: 368



## 4. Dataset class y collate function

**Fix #1 (bug original):** la versión vieja de `collate_pair_data` calculaba `max_len` y el padding mirando **solo** las coordenadas INACTIVE de cada muestra del batch, y nunca igualaba `active_coords`/`active_adj` a ese mismo largo. Como INACTIVE y ACTIVE de una misma kinasa pueden tener distinta cantidad de residuos, esto producía tensores con tamaños distintos dentro del mismo batch.

**Fix #2 (bug encontrado al correr con el dataset real):** el embedding por residuo (`sample['embedding']`) puede tener una cantidad de filas **distinta** a `ca_coords` para la misma muestra (no necesariamente coinciden 1 a 1). Eso provocaba que, aun con `inactive_coords` del mismo tamaño entre dos muestras del batch, sus `inactive_embed` tuvieran tamaños distintos (ej. 529 filas vs 314 filas) y el `torch.stack` final fallara.

El fix para ambos: en `ConformationPairDataset.__getitem__` alineamos explícitamente **tanto** `active_coords` **como** `inactive_embed` al mismo N que `inactive_coords` (truncando o paddeando), ya que el modelo predice coordenadas ACTIVE a partir de la estructura INACTIVE nodo a nodo — la salida del modelo siempre tiene N = N_inactive, y el embedding que entra al modelo también debe tener ese N. El collate además incluye un chequeo de seguridad por si algún sample llegara desalineado.

In [7]:
class ConformationPairDataset(Dataset):
    """Dataset de pares ACTIVE/INACTIVE para una misma kinasa."""

    def __init__(self, pairs_list):
        self.pairs = pairs_list

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        inactive = pair['inactive']
        active = pair['active']

        inactive_coords = np.array(inactive['ca_coords'], dtype=np.float32)
        active_coords = np.array(active['ca_coords'], dtype=np.float32)

        inactive_embed = np.array(inactive['embedding'], dtype=np.float32)
        if inactive_embed.ndim == 3:
            inactive_embed = np.mean(inactive_embed, axis=1)

        inactive_adj = np.array(inactive['adjacency'], dtype=np.float32)

        # IMPORTANTE: el modelo predice coordenadas ACTIVE a partir de la
        # estructura INACTIVE, nodo a nodo. La salida del modelo siempre
        # tendrá N = N_inactive. Por eso alineamos ACTIVE y el EMBEDDING a ese
        # mismo N aquí, en el Dataset (fix del bug original con active_coords,
        # + fix de embedding con cantidad de filas distinta a ca_coords, que
        # puede pasar según cómo se generó el embedding por residuo).
        n_inactive = len(inactive_coords)
        n_active = len(active_coords)

        n_embed = len(inactive_embed)
        if n_embed >= n_inactive:
            inactive_embed = inactive_embed[:n_inactive]
        else:
            pad_emb = n_inactive - n_embed
            inactive_embed = np.pad(inactive_embed, ((0, pad_emb), (0, 0)), constant_values=0)

        if n_active >= n_inactive:
            active_coords = active_coords[:n_inactive]
        else:
            pad_len = n_inactive - n_active
            active_coords = np.pad(active_coords, ((0, pad_len), (0, 0)), constant_values=0)

        valid_mask = np.ones(n_inactive, dtype=np.float32)
        # Si ACTIVE era más corto, esas posiciones no son residuos reales: no penalizar la loss ahí
        if n_active < n_inactive:
            valid_mask[n_active:] = 0.0

        return {
            'inactive_coords': torch.from_numpy(inactive_coords),
            'inactive_embed': torch.from_numpy(inactive_embed),
            'inactive_adj': torch.from_numpy(inactive_adj),
            'active_coords': torch.from_numpy(active_coords),
            'valid_mask': torch.from_numpy(valid_mask),
            'kinase': pair['kinase']
        }


def collate_pair_data(batch):
    """
    Hace pad de cada muestra del batch al N máximo de INACTIVE del batch.
    Como ConformationPairDataset ya garantiza que active_coords, inactive_embed
    y valid_mask tienen el mismo N que inactive_coords para cada muestra
    individual, acá sólo hace falta igualar tamaños ENTRE muestras del batch.
    """
    def to_tensor(x, dtype=torch.float32):
        return x if isinstance(x, torch.Tensor) else torch.from_numpy(np.array(x)).to(dtype)

    items = []
    max_len = 0
    emb_width = 0
    for item in batch:
        ic = to_tensor(item['inactive_coords'])
        ie = to_tensor(item['inactive_embed'])
        if ie.dim() == 1:
            ie = ie.unsqueeze(1)
        # Sanity check: en este punto ie y ic deberían tener el mismo N
        # (lo garantiza ConformationPairDataset). Si por algún motivo no
        # coinciden, lo forzamos acá para que el batch nunca rompa.
        if ie.shape[0] != ic.shape[0]:
            n = ic.shape[0]
            if ie.shape[0] > n:
                ie = ie[:n]
            else:
                pad = torch.zeros((n - ie.shape[0], ie.shape[1]), dtype=ie.dtype)
                ie = torch.cat([ie, pad], dim=0)
        items.append({
            'inactive_coords': ic,
            'inactive_embed': ie,
            'inactive_adj': to_tensor(item['inactive_adj']),
            'active_coords': to_tensor(item['active_coords']),
            'valid_mask': to_tensor(item['valid_mask']),
            'kinase': item.get('kinase')
        })
        max_len = max(max_len, ic.shape[0])
        emb_width = max(emb_width, ie.shape[1])

    for it in items:
        n = it['inactive_coords'].shape[0]
        # Pad de embedding en columnas
        if it['inactive_embed'].shape[1] < emb_width:
            pad_cols = torch.zeros((n, emb_width - it['inactive_embed'].shape[1]))
            it['inactive_embed'] = torch.cat([it['inactive_embed'], pad_cols], dim=1)

        if n < max_len:
            pad_rows = max_len - n
            it['inactive_coords'] = torch.cat([it['inactive_coords'], torch.zeros((pad_rows, 3))], dim=0)
            it['active_coords'] = torch.cat([it['active_coords'], torch.zeros((pad_rows, 3))], dim=0)
            it['inactive_embed'] = torch.cat([it['inactive_embed'], torch.zeros((pad_rows, emb_width))], dim=0)
            it['valid_mask'] = torch.cat([it['valid_mask'], torch.zeros(pad_rows)], dim=0)

            old_adj = it['inactive_adj']
            new_adj = torch.zeros((max_len, max_len), dtype=old_adj.dtype)
            new_adj[:n, :n] = old_adj
            it['inactive_adj'] = new_adj

    return {
        'inactive_coords': torch.stack([it['inactive_coords'] for it in items]),
        'inactive_embed': torch.stack([it['inactive_embed'] for it in items]),
        'inactive_adj': torch.stack([it['inactive_adj'] for it in items]),
        'active_coords': torch.stack([it['active_coords'] for it in items]),
        'valid_mask': torch.stack([it['valid_mask'] for it in items]),
        'kinase': [it['kinase'] for it in items]
    }


print("✓ ConformationPairDataset y collate_pair_data (corregidos) definidos")


✓ ConformationPairDataset y collate_pair_data (corregidos) definidos


## 5. Modelo EGNN (versión sparse, optimizada para CPU)

**Por qué esta versión:** la matriz `adjacency` real es muy sparse (en tus datos: densidad ~5.8%, cada residuo conectado a ~17 de sus ~300 vecinos posibles — son contactos de cadena/3D, no todos-contra-todos). La implementación original construía un tensor denso `(N, N, ...)` para *todos* los pares de residuos y después multiplicaba por la máscara `adj` — es decir, calculaba y tiraba a la basura ~94% del trabajo. Eso explica por qué una época tardaba ~16 minutos: con N≈300 son ~90.000 pares calculados cuando sólo ~5.000 son edges reales.

Esta versión usa una representación **edge-list** (`torch.nonzero` sobre `adj` para obtener los índices de los edges existentes, y `index_add_` para agregar mensajes por nodo) — matemáticamente equivalente a la versión densa (validado numéricamente, diferencia ~1e-7 por redondeo de floats), pero solo opera sobre los edges que existen. En nuestras pruebas con N=297 y densidad 5.8%, esto fue ~37x más rápido (4.95s → 0.13s por forward+backward de las 3 capas).

**Trade-off:** esta versión sólo soporta `batch_size=1` (procesa un grafo/proteína por vez), por eso fijamos `BATCH_SIZE = 1` en la Sección de configuración. Como tus proteínas varían bastante en tamaño (234 a 614 residuos), de todas formas el padding de un batch>1 ya desperdiciaba cómputo — así que no se pierde nada relevante.

In [8]:
class EGNNLayer(nn.Module):
    """
    Versión sparse (edge-list) del EGNNLayer, equivalente matemáticamente a la
    versión densa pero O(E) en vez de O(N^2), donde E es la cantidad de edges
    reales en `adj` (validado numéricamente contra la versión densa).
    Procesa un solo grafo por vez (B debe ser 1).
    """
    def __init__(self, node_dim, hidden_dim=128):
        super().__init__()
        self.node_dim = node_dim
        self.hidden_dim = hidden_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * node_dim + 1, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(node_dim + hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, node_dim)
        )
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1)
        )

    def forward(self, h, coords, adj):
        # h: (1,N,D), coords: (1,N,3), adj: (1,N,N)  — batch_size fijo en 1
        B, N, D = h.shape
        assert B == 1, "EGNNLayer sparse requiere batch_size=1"
        h0, coords0, adj0 = h[0], coords[0], adj[0]

        src, dst = torch.nonzero(adj0, as_tuple=True)  # índices de los edges existentes
        if src.numel() == 0:
            return h, coords  # sin edges (caso degenerado): no hay nada que propagar

        coord_diff_e = coords0[src] - coords0[dst]                            # (E,3)
        dist_e = torch.sqrt((coord_diff_e ** 2).sum(-1, keepdim=True) + 1e-8)  # (E,1)

        h_i = h0[src]  # (E,D)
        h_j = h0[dst]  # (E,D)
        edge_input = torch.cat([h_i, h_j, dist_e], dim=-1)  # (E, 2D+1)

        e = self.edge_mlp(edge_input)  # (E, hidden)

        # Agregar mensajes de cada edge al nodo de origen (equivalente a e.sum(2) en la version densa)
        agg = torch.zeros(N, e.shape[-1], device=h0.device, dtype=e.dtype)
        agg.index_add_(0, src, e)

        h_new = self.node_mlp(torch.cat([h0, agg], dim=-1))  # (N,D)

        coord_messages_e = self.coord_mlp(e)  # (E,1)
        coord_update = torch.zeros(N, 3, device=h0.device, dtype=coords0.dtype)
        coord_update.index_add_(0, src, coord_messages_e * coord_diff_e)  # (N,3)

        h_out = (h0 + h_new).unsqueeze(0)
        coords_out = (coords0 + coord_update).unsqueeze(0)
        return h_out, coords_out


class SimpleEGNN(nn.Module):
    def __init__(self, embedding_dim=1280, node_dim=128, num_layers=3, hidden_dim=128):
        super().__init__()
        self.node_proj = nn.Linear(embedding_dim, node_dim)
        self.layers = nn.ModuleList([EGNNLayer(node_dim=node_dim, hidden_dim=hidden_dim) for _ in range(num_layers)])
        self.coord_out = nn.Sequential(
            nn.Linear(node_dim, node_dim), nn.ReLU(), nn.Linear(node_dim, 3)
        )

    def forward(self, node_features, coords, adj):
        h = self.node_proj(node_features)
        for layer in self.layers:
            h, coords = layer(h, coords, adj)
        offsets = self.coord_out(h)
        return coords + offsets


print("✓ EGNNLayer (sparse) y SimpleEGNN definidos")


✓ EGNNLayer (sparse) y SimpleEGNN definidos


## 6. Loss, métricas, entrenamiento y validación

In [ ]:
def masked_mse_loss(pred, target, mask):
    # pred/target: (B,N,3), mask: (B,N)
    se = ((pred - target) ** 2).sum(-1)  # (B,N)
    masked_se = se * mask
    denom = mask.sum(dim=1).clamp_min(1.0)
    loss_per_sample = masked_se.sum(dim=1) / denom
    return loss_per_sample.mean()


def kabsch_align(P, Q, mask):
    """
    Rigid body alignment (Kabsch) of Q to P using only valid atoms (mask > 0.5).
    
    Args:
        P: predicted coordinates, shape (B, N, 3)
        Q: target coordinates, shape (B, N, 3)
        mask: valid mask, shape (B, N), where 1 = valid, 0 = padding
    
    Returns:
        Q_aligned: Q rigidly transformed (rotated + translated) to align with P,
                   shape (B, N, 3). Transformation computed only from valid atoms,
                   but applied to all positions (preserving padding structure).
    """
    B, N, D = P.shape
    assert D == 3, "Coordinates must be 3D"
    
    Q_aligned_list = []
    
    for b in range(B):
        valid_idx = (mask[b] > 0.5).nonzero(as_tuple=True)[0]
        
        if len(valid_idx) < 3:
            # Not enough atoms to compute rotation; return Q unchanged
            Q_aligned_list.append(Q[b])
            continue
        
        P_valid = P[b, valid_idx]  # (n_valid, 3)
        Q_valid = Q[b, valid_idx]  # (n_valid, 3)
        
        # Center both point clouds
        P_mean = P_valid.mean(dim=0, keepdim=True)  # (1, 3)
        Q_mean = Q_valid.mean(dim=0, keepdim=True)  # (1, 3)
        
        P_centered = P_valid - P_mean  # (n_valid, 3)
        Q_centered = Q_valid - Q_mean  # (n_valid, 3)
        
        # Compute covariance matrix H = Q^T * P
        H = torch.mm(Q_centered.T, P_centered)  # (3, 3)
        
        # SVD
        U, S, Vt = torch.linalg.svd(H)
        
        # Compute rotation matrix R with reflection correction via a
        # diagonal sign-correction matrix. NOTE: we deliberately avoid
        # mutating Vt/U in-place (e.g. Vt[-1, :] *= -1) because both are
        # direct outputs of torch.linalg.svd and participate in the
        # autograd graph (LinalgSvdBackward0). An in-place op on them
        # invalidates the saved tensors the SVD backward needs, causing
        # "one of the variables needed for gradient computation has been
        # modified by an inplace operation" at loss.backward(). Building
        # a fresh diagonal matrix D keeps everything out-of-place.
        det_sign = torch.sign(torch.det(torch.mm(Vt.T, U.T)))
        D = torch.diag(torch.tensor([1.0, 1.0, det_sign], device=H.device, dtype=H.dtype))
        R = torch.mm(torch.mm(Vt.T, D), U.T)  # (3, 3)
        
        # Compute translation (after rotation)
        t = P_mean - torch.mm(Q_mean, R.T) # (1, 3)
        
        # Apply transformation to all coordinates (including padding)
        Q_rot = torch.mm(Q[b], R.T)  # (N, 3)
        Q_aligned = Q_rot + t  # (N, 3)
        
        Q_aligned_list.append(Q_aligned)
    
    return torch.stack(Q_aligned_list, dim=0)  # (B, N, 3)


def compute_metrics_batch(predicted_coords, target_coords, valid_mask):
    batch_metrics = {'rmsd': [], 'mae': [], 'dist_error': [], 'recovery': []}
    pred_np = predicted_coords.detach().cpu().numpy()
    tgt_np = target_coords.detach().cpu().numpy()
    mask_np = valid_mask.detach().cpu().numpy()

    for b in range(pred_np.shape[0]):
        valid_idx = mask_np[b] > 0.5
        if valid_idx.sum() < 2:
            continue
        p, t = pred_np[b][valid_idx], tgt_np[b][valid_idx]
        diff = p - t
        rmsd_val = np.sqrt(np.mean((diff ** 2).sum(axis=1)))
        mae_val = np.mean(np.abs(diff))
        dp = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1))
        dt = np.sqrt(((t[:, None, :] - t[None, :, :]) ** 2).sum(-1))
        dist_err = np.mean(np.abs(dp - dt))
        recovery = 100.0 * np.sum(np.sqrt((diff ** 2).sum(axis=1)) <= COORD_THRESHOLD) / len(diff)
        batch_metrics['rmsd'].append(rmsd_val)
        batch_metrics['mae'].append(mae_val)
        batch_metrics['dist_error'].append(dist_err)
        batch_metrics['recovery'].append(recovery)

    return {k: (np.mean(v) if len(v) > 0 else 0.0) for k, v in batch_metrics.items()}


def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    all_metrics = {'rmsd': [], 'mae': [], 'dist_error': [], 'recovery': []}
    n_batches = 0

    for batch in dataloader:
        inactive_coords = batch['inactive_coords'].to(device).float()
        inactive_embed = batch['inactive_embed'].to(device).float()
        inactive_adj = batch['inactive_adj'].to(device).float()
        active_coords = batch['active_coords'].to(device).float()
        valid_mask = batch['valid_mask'].to(device).float()

        optimizer.zero_grad()
        predicted_coords = model(inactive_embed, inactive_coords, inactive_adj)
        
        # *** KABSCH: Align target to prediction ***
        active_coords_aligned = kabsch_align(predicted_coords, active_coords, valid_mask)
        
        loss = masked_mse_loss(predicted_coords, active_coords_aligned, valid_mask)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1
        # Metrics computed on aligned coordinates
        metrics = compute_metrics_batch(predicted_coords, active_coords_aligned, valid_mask)
        for key in all_metrics:
            all_metrics[key].append(metrics[key])

    avg_metrics = {k: (np.mean(v) if len(v) > 0 else 0.0) for k, v in all_metrics.items()}
    return (total_loss / max(1, n_batches)), avg_metrics


def validate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_metrics = {'rmsd': [], 'mae': [], 'dist_error': [], 'recovery': []}
    n_batches = 0

    with torch.no_grad():
        for batch in dataloader:
            inactive_coords = batch['inactive_coords'].to(device).float()
            inactive_embed = batch['inactive_embed'].to(device).float()
            inactive_adj = batch['inactive_adj'].to(device).float()
            active_coords = batch['active_coords'].to(device).float()
            valid_mask = batch['valid_mask'].to(device).float()

            predicted_coords = model(inactive_embed, inactive_coords, inactive_adj)
            
            # *** KABSCH: Align target to prediction ***
            active_coords_aligned = kabsch_align(predicted_coords, active_coords, valid_mask)
            
            loss = masked_mse_loss(predicted_coords, active_coords_aligned, valid_mask)
            total_loss += loss.item()
            n_batches += 1
            # Metrics computed on aligned coordinates
            metrics = compute_metrics_batch(predicted_coords, active_coords_aligned, valid_mask)
            for key in all_metrics:
                all_metrics[key].append(metrics[key])

    avg_metrics = {k: (np.mean(v) if len(v) > 0 else 0.0) for k, v in all_metrics.items()}
    return (total_loss / max(1, n_batches)), avg_metrics


print("✓ Funciones de loss, métricas, train_epoch y validate definidas")
print("✓ Kabsch rigid-body alignment ENABLED for all loss/metrics calculations")


## 7. Cargar folds LOKO y preparar datasets por fold

In [13]:
print("Loading LOKO folds...")
folds_dir = Path('data/loko_folds')
folds = {}

if folds_dir.exists():
    for fold_file in sorted(folds_dir.glob('fold_*.pkl')):
        with open(fold_file, 'rb') as f:
            folds[fold_file.stem] = pickle.load(f)
    print(f"✓ Loaded {len(folds)} LOKO folds")
else:
    print(f"⚠️  LOKO folds directory not found at {folds_dir}")


def prepare_fold_datasets(fold_data):
    test_kinase = fold_data.get('test_kinase') if isinstance(fold_data, dict) else None
    train_kinases = set()

    if isinstance(fold_data, dict) and 'train_dataset' in fold_data and isinstance(fold_data['train_dataset'], dict):
        for s in fold_data['train_dataset'].get('structures', []):
            if isinstance(s, dict) and s.get('kinase_name') is not None:
                train_kinases.add(s['kinase_name'])
    if not train_kinases and isinstance(fold_data, dict) and 'train_kinases' in fold_data:
        train_kinases.update(fold_data['train_kinases'])

    train_pairs = []
    for kinase in train_kinases:
        train_pairs.extend(all_pairs.get(kinase, []))

    test_pairs = all_pairs.get(test_kinase, [])

    val_pairs = []
    if train_pairs:
        np.random.shuffle(train_pairs)
        split_idx = int(0.8 * len(train_pairs))
        train_pairs, val_pairs = train_pairs[:split_idx], train_pairs[split_idx:]

    return {
        'train_pairs': train_pairs,
        'val_pairs': val_pairs,
        'test_pairs': test_pairs,
        'train_kinases': sorted(train_kinases),
        'test_kinase': test_kinase
    }


loko_datasets = {}
if folds:
    for fold_name, fold_data in sorted(folds.items()):
        loko_datasets[fold_name] = prepare_fold_datasets(fold_data)
    print(f"✓ Prepared {len(loko_datasets)} LOKO folds")
else:
    print("⚠️  'folds' está vacío — no se pudo preparar loko_datasets")


Loading LOKO folds...
✓ Loaded 6 LOKO folds
✓ Prepared 6 LOKO folds


## 8. Loop de entrenamiento y evaluación LOKO

In [14]:
figures_dir = Path('figures/egnn_generation')
figures_dir.mkdir(parents=True, exist_ok=True)

# CAMBIO A: Crear directorio para checkpoints
checkpoints_dir = Path('checkpoints/egnn_loko')
checkpoints_dir.mkdir(parents=True, exist_ok=True)

results_per_fold = {}

for fold_name in sorted(loko_datasets.keys()):
    fold_info = loko_datasets[fold_name]
    print(f"\n=== Fold {fold_name} | test kinase: {fold_info['test_kinase']} ===")

    train_pairs = fold_info['train_pairs']
    val_pairs = fold_info['val_pairs']
    test_pairs = fold_info['test_pairs']

    if len(train_pairs) == 0:
        print('  ⚠️  No training pairs, skipping')
        continue

    train_dataset = ConformationPairDataset(train_pairs)
    val_dataset = ConformationPairDataset(val_pairs) if val_pairs else None
    test_dataset = ConformationPairDataset(test_pairs) if test_pairs else None

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_pair_data)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pair_data) if val_dataset else None
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pair_data) if test_dataset else None

    # Inferir dimensión del embedding desde una muestra de entrenamiento
    def infer_embedding_dim(pair):
        emb = pair['inactive'].get('embedding')
        if emb is None:
            return 1280
        arr = np.array(emb)
        if arr.ndim == 1:
            return arr.shape[0]
        if arr.ndim == 2:
            return arr.shape[1]
        if arr.ndim == 3:
            return arr.shape[2]
        return 1280

    emb_dim = infer_embedding_dim(train_pairs[0]) if train_pairs else 1280

    model = SimpleEGNN(embedding_dim=emb_dim, node_dim=128, num_layers=3, hidden_dim=128).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_loss = float('inf')
    best_state = None
    train_losses, val_losses = [], []

    for epoch in range(1, EPOCHS + 1):
        train_loss, _ = train_epoch(model, train_loader, optimizer, DEVICE)
        train_losses.append(train_loss)

        if val_loader:
            val_loss, _ = validate(model, val_loader, DEVICE)
            val_losses.append(val_loss)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            val_loss = 0.0

        if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
            print(f"  Epoch {epoch}/{EPOCHS}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    print('  ✓ Training finished')

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    if test_loader is not None:
        test_loss, test_metrics = validate(model, test_loader, DEVICE)
        print('  Test metrics (post-Kabsch alignment):')
        print(f"    RMSD: {test_metrics['rmsd']:.4f} Å | MAE: {test_metrics['mae']:.4f} Å | "
              f"DistErr: {test_metrics['dist_error']:.4f} Å | Recovery: {test_metrics['recovery']:.2f}%")
    else:
        test_metrics = {'rmsd': 0, 'mae': 0, 'dist_error': 0, 'recovery': 0}
        print('  ⚠️  No test pairs for this fold')

    results_per_fold[fold_name] = {
        'test_kinase': fold_info['test_kinase'],
        'train_kinases': fold_info['train_kinases'],
        'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
        'train_losses': train_losses,
        'val_losses': val_losses,
        'test_metrics': test_metrics,
        'test_pairs': test_pairs
    }

    # CAMBIO A: Guardar checkpoint a disco
    checkpoint_path = checkpoints_dir / f"{fold_name}.pt"
    torch.save({
        'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
        'emb_dim': emb_dim,
        'fold_name': fold_name,
        'test_kinase': fold_info['test_kinase'],
        'train_kinases': fold_info['train_kinases'],
        'train_losses': train_losses,
        'val_losses': val_losses,
        'test_metrics': test_metrics,
        'best_val_loss': best_val_loss,
    }, checkpoint_path)
    print(f'  ✓ Checkpoint saved: {checkpoint_path}')

    # Visualización: hasta 3 pares de test, ACTIVE real vs. PRED en subplots
    # lado a lado (en vez de overlay en el mismo eje, que era difícil de leer).
    # Reusamos ConformationPairDataset para obtener inactive_embed ya alineado
    # a inactive_coords (mismo fix que en el entrenamiento) en vez de leer
    # 'embedding' crudo del dict, que puede tener distinta cantidad de filas.
    n_vis = min(3, len(test_pairs))
    vis_dataset = ConformationPairDataset(test_pairs[:n_vis])
    for i in range(n_vis):
        pair = test_pairs[i]
        sample = vis_dataset[i]

        inactive_coords = sample['inactive_coords'].numpy()
        inactive_embed = sample['inactive_embed'].numpy()
        inactive_adj = sample['inactive_adj'].numpy()

        with torch.no_grad():
            inp_embed = torch.from_numpy(inactive_embed).unsqueeze(0).to(DEVICE).float()
            inp_coords = torch.from_numpy(inactive_coords).unsqueeze(0).to(DEVICE).float()
            inp_adj = torch.from_numpy(inactive_adj).unsqueeze(0).to(DEVICE).float()
            pred_coords_t = model(inp_embed, inp_coords, inp_adj)

            # *** KABSCH: alinear ACTIVE real (target) a la predicción, igual ***
            # *** que en train_epoch/validate, para que el RMSD acá coincida ***
            # *** con test_metrics['rmsd'] reportado por consola.            ***
            true_coords_t = sample['active_coords'].unsqueeze(0).to(DEVICE).float()
            valid_mask_t = sample['valid_mask'].unsqueeze(0).to(DEVICE).float()
            true_coords_aligned_t = kabsch_align(pred_coords_t, true_coords_t, valid_mask_t)

            pred_coords = pred_coords_t.squeeze(0).cpu().numpy()
            true_coords = true_coords_aligned_t.squeeze(0).cpu().numpy()

        # usamos valid_mask para no graficar posiciones de padding
        mask = sample['valid_mask'].numpy() > 0.5
        true_trim = true_coords[mask]
        pred_trim = pred_coords[mask]

        d = pred_trim - true_trim
        rmsd_val = np.sqrt(np.mean((d ** 2).sum(axis=1)))

        fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': '3d'})

        # Límites de ejes combinados para que ambos subplots sean comparables
        all_coords = np.concatenate([true_trim, pred_trim], axis=0)
        x_min, x_max = all_coords[:, 0].min(), all_coords[:, 0].max()
        y_min, y_max = all_coords[:, 1].min(), all_coords[:, 1].max()
        z_min, z_max = all_coords[:, 2].min(), all_coords[:, 2].max()

        axes[0].plot(true_trim[:, 0], true_trim[:, 1], true_trim[:, 2], color='green')
        axes[0].scatter(true_trim[:, 0], true_trim[:, 1], true_trim[:, 2], c='green', s=20)
        axes[0].set_title('ACTIVE (real)')
        axes[0].set_xlim(x_min, x_max)
        axes[0].set_ylim(y_min, y_max)
        axes[0].set_zlim(z_min, z_max)
        axes[0].view_init(elev=20, azim=45)

        axes[1].plot(pred_trim[:, 0], pred_trim[:, 1], pred_trim[:, 2], color='red', linestyle='--')
        axes[1].scatter(pred_trim[:, 0], pred_trim[:, 1], pred_trim[:, 2], c='red', s=20)
        axes[1].set_title(f'PRED (RMSD={rmsd_val:.2f}Å)')
        axes[1].set_xlim(x_min, x_max)
        axes[1].set_ylim(y_min, y_max)
        axes[1].set_zlim(z_min, z_max)
        axes[1].view_init(elev=20, azim=45)

        fig.suptitle(f"{fold_name} - {pair['kinase']} - sample {i} | RMSD {rmsd_val:.2f} Å")
        fig.savefig(figures_dir / f"{fold_name}_sample{i}_overlay.png")
        plt.close(fig)

print('\n=== LOKO training complete ===')

rows = []
for fold, info in results_per_fold.items():
    tm = info['test_metrics']
    rows.append({'fold': fold, 'test_kinase': info['test_kinase'], 'rmsd': tm['rmsd'],
                 'mae': tm['mae'], 'dist_err': tm['dist_error'], 'recovery': tm['recovery']})

results_df = pd.DataFrame(rows)
results_df.to_csv(figures_dir / 'loko_results_summary.csv', index=False)

# CAMBIO A: Guardar resumen de resultados en pickle
summary_to_save = {
    'results_df': results_df,
    'checkpoints_dir': str(checkpoints_dir),
    'folds_trained': list(results_per_fold.keys()),
}
with open(checkpoints_dir / 'all_results_summary.pkl', 'wb') as f:
    pickle.dump(summary_to_save, f)
print(f'✓ Results summary saved: {checkpoints_dir / "all_results_summary.pkl"}')

print(results_df)



=== Fold fold_01 | test kinase: ABL1 ===


RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [3, 3]], which is output 2 of LinalgSvdBackward0, is at version 2; expected version 0 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True, check_nan=False).

## 9. Resumen científico

In [ ]:
print("=" * 80)
print("SCIENTIFIC SUMMARY")
print("=" * 80)

print(f"\n### Dataset Overview ###")
print(f"  Total samples: {len(dataset)}")
print(f"  Unique kinases: {len(kinase_samples)}")

print(f"\n### Pair Generation ###")
print(f"  Total pairs generated: {total_pairs}")

print(f"\n### Model Architecture ###")
print(f"  EGNN layers: 3 | Node dimension: 128 | Hidden dimension: 128")

print("\n### LOKO Results (per fold) ###")
print("NOTE: All metrics below are POST-KABSCH alignment (rotation/translation invariant)")
print("      Previous baseline (without Kabsch): RMSD ~12.54 Å, Recovery ~1.40%")
print(results_df.to_string(index=False))

if len(results_df) > 0:
    print(f"\n### LOKO Results (promedio across folds) ###")
    print(f"  RMSD (Å) [post-Kabsch]: {results_df['rmsd'].mean():.4f}")
    print(f"  MAE (Å) [post-Kabsch]: {results_df['mae'].mean():.4f}")
    print(f"  Distance Matrix Error (Å) [post-Kabsch]: {results_df['dist_err'].mean():.4f}")
    print(f"  Structural Recovery (≤{COORD_THRESHOLD}Å) [post-Kabsch]: {results_df['recovery'].mean():.2f}%")

print(f"\n### Visualizations Generated ###")
print(f"  ✓ {figures_dir}/<fold>_sample<i>_overlay.png")
print(f"  ✓ {figures_dir}/loko_results_summary.csv")

print("\n" + "=" * 80)
print('✓ Analysis complete!')
print("=" * 80)


SCIENTIFIC SUMMARY

### Dataset Overview ###
  Total samples: 797
  Unique kinases: 6

### Pair Generation ###
  Total pairs generated: 368

### Model Architecture ###
  EGNN layers: 3 | Node dimension: 128 | Hidden dimension: 128

### LOKO Results (per fold) ###
   fold test_kinase      rmsd      mae  dist_err  recovery
fold_01        ABL1 12.625774 5.701236  6.936458  1.098736
fold_02        BRAF 10.780725 4.837099  6.483495  1.076606
fold_03        EGFR 16.952255 7.019691  7.937948  0.717794
fold_04       FGFR1 10.899872 4.982963  5.876103  1.429063
fold_05         KIT 10.914225 4.810702  5.873572  2.644017
fold_06      PDGFRA 13.073665 5.677630  6.783134  1.428181

### LOKO Results (promedio across folds) ###
  RMSD (Å): 12.5411
  MAE (Å): 5.5049
  Distance Matrix Error (Å): 6.6485
  Structural Recovery (≤2.0Å): 1.40%

### Visualizations Generated ###
  ✓ figures/egnn_generation/<fold>_sample<i>_overlay.png
  ✓ figures/egnn_generation/loko_results_summary.csv

✓ Analysis complete!


## 10. Verificación de equivariancia rotacional

In [ ]:
def test_equivariance(model, inactive_embed, inactive_coords, inactive_adj, device, seed=42):
    """
    Test rotational equivariance: rotate input coordinates and verify output rotates similarly.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Forward pass on original coordinates
    with torch.no_grad():
        embed_tensor = torch.from_numpy(inactive_embed).unsqueeze(0).to(device).float()
        coord_tensor = torch.from_numpy(inactive_coords).unsqueeze(0).to(device).float()
        adj_tensor = torch.from_numpy(inactive_adj).unsqueeze(0).to(device).float()
        output_orig = model(embed_tensor, coord_tensor, adj_tensor)  # shape: (1, N, 3)
    
    # Random rotation matrix
    u, _ = np.linalg.qr(np.random.randn(3, 3))
    det = np.linalg.det(u)
    if det < 0:
        u[:, -1] *= -1
    
    # Apply rotation to input coordinates
    coord_rotated = np.dot(inactive_coords, u.T)  # (N, 3)
    
    # Forward pass on rotated coordinates
    with torch.no_grad():
        coord_rot_tensor = torch.from_numpy(coord_rotated).unsqueeze(0).to(device).float()
        output_rot = model(embed_tensor, coord_rot_tensor, adj_tensor)
    
    # Extract predictions (both batch_size=1)
    pred_orig = output_orig[0].cpu().numpy()  # (N, 3)
    pred_rot = output_rot[0].cpu().numpy()
    
    # Apply same rotation to original output (expected)
    pred_orig_rotated = np.dot(pred_orig, u.T)
    
    # Compare: output_rot should ≈ pred_orig_rotated
    diff = pred_rot - pred_orig_rotated  # (N, 3)
    max_diff = np.sqrt((diff ** 2).sum(axis=1)).max()
    output_norm = np.sqrt((pred_orig ** 2).sum(axis=1)).max()
    mean_rel_error = np.mean(np.abs(diff)) / max(output_norm, 1e-8)
    
    is_equivariant = max_diff < 1e-3
    
    return {
        'max_diff': max_diff,
        'mean_rel_error': mean_rel_error,
        'is_equivariant': is_equivariant
    }


# Test on 3 examples from different folds if possible
print("\n" + "=" * 80)
print("ROTATIONAL EQUIVARIANCE TEST")
print("=" * 80)

# Priority 1: Use loaded_models from Sección 11
# Priority 2: Use results_per_fold from Sección 8
# Priority 3: Neither (inform user and skip)

models_to_test = {}
models_source = None

if 'loaded_models' in locals() and len(loaded_models) > 0:
    models_to_test = loaded_models
    models_source = "LOADED FROM CHECKPOINTS (Sección 11)"
    print(f"\n✓ Using models from loaded_models (Sección 11)")
elif 'results_per_fold' in locals() and len(results_per_fold) > 0:
    models_source = "FROM TRAINING SESSION (Sección 8)"
    print(f"\n✓ Using models from results_per_fold (Sección 8 training session)")
    
    fold_list = sorted(results_per_fold.keys())
    for fold_idx, fold_name in enumerate(fold_list[:3]):
        fold_result = results_per_fold[fold_name]
        test_pairs_fold = fold_result.get('test_pairs', [])
        
        if len(test_pairs_fold) == 0:
            continue
        
        # Recreate model
        emb_dim_test = infer_embedding_dim(test_pairs_fold[0]) if test_pairs_fold else 1280
        model_temp = SimpleEGNN(embedding_dim=emb_dim_test, node_dim=128, num_layers=3, hidden_dim=128).to(DEVICE)
        model_temp.load_state_dict({k: v.to(DEVICE) for k, v in fold_result['model_state_dict'].items()})
        model_temp.eval()
        
        models_to_test[fold_name] = {
            'model': model_temp,
            'test_kinase': fold_result.get('test_kinase', 'unknown'),
            'test_pairs': test_pairs_fold,
        }
else:
    print("\n⚠️  No models found. Please:")
    print("   1. Run Sección 8 (training) to train models, OR")
    print("   2. Run Sección 11 (reload checkpoints) to load existing models")
    models_to_test = {}

equivariance_results = []

fold_list = sorted(models_to_test.keys())[:3]
for fold_idx, fold_name in enumerate(fold_list):
    model_info = models_to_test[fold_name]
    model_test = model_info['model']
    test_kinase = model_info.get('test_kinase', 'unknown')
    
    # Get test pairs if available
    if 'test_pairs' in model_info:
        test_pairs_fold = model_info['test_pairs']
    else:
        # If loaded from checkpoints (Sección 11), test_pairs are not stored
        # (too large to persist). Skip per-sample testing.
        print(f"⊘ {fold_name} ({test_kinase}): skipping (test_pairs not available)")
        continue
    
    if len(test_pairs_fold) == 0:
        continue
    
    # Use first test sample
    pair = test_pairs_fold[0]
    sample_dataset = ConformationPairDataset([pair])
    sample = sample_dataset[0]
    
    inactive_coords = sample['inactive_coords'].numpy()
    inactive_embed = sample['inactive_embed'].numpy()
    inactive_adj = sample['inactive_adj'].numpy()
    
    # Run test with a unique seed per fold
    result = test_equivariance(model_test, inactive_embed, inactive_coords, inactive_adj, DEVICE, seed=100+fold_idx)
    
    equivariance_results.append({
        'fold': fold_name,
        'kinase': test_kinase,
        'max_diff': result['max_diff'],
        'mean_rel_error': result['mean_rel_error'],
        'equivariant_ok': result['is_equivariant']
    })
    
    # Print result
    if result['is_equivariant']:
        print(f"✓ {fold_name} ({test_kinase}): equivariante (max_diff={result['max_diff']:.2e})")
    else:
        print(f"⚠️  {fold_name} ({test_kinase}): NO equivariante (max_diff={result['max_diff']:.2e})")
        print(f"   → Revisar EGNNLayer.coord_mlp: el update de coordenadas debe construirse")
        print(f"   → como combinación de vectores coord_diff (coords[i]-coords[j]), nunca")
        print(f"   → usando coordenadas absolutas como input directo a una MLP.")

# Summary table
if equivariance_results:
    equiv_df = pd.DataFrame(equivariance_results)
    print("\n" + "=" * 80)
    print("EQUIVARIANCE SUMMARY TABLE")
    print("=" * 80)
    print(equiv_df.to_string(index=False))
    
    n_equivariant = equiv_df['equivariant_ok'].sum()
    n_total = len(equiv_df)
    print(f"\n✓ Equivariant models: {n_equivariant}/{n_total}")
    print(f"✓ Source: {models_source}")
else:
    print("\n⚠️  No equivariance tests completed.")



ROTATIONAL EQUIVARIANCE TEST


NameError: name 'results_per_fold' is not defined

## 11. Recargar modelos entrenados (sin re-entrenar)

In [ ]:
# CAMBIO B: Recargar modelos entrenados desde checkpoints
print("\n" + "=" * 80)
print("SECTION 11: RELOAD TRAINED MODELS FROM CHECKPOINTS")
print("=" * 80)

checkpoints_dir = Path('checkpoints/egnn_loko')

if not checkpoints_dir.exists():
    print(f"\n⚠️  Checkpoint directory not found: {checkpoints_dir}")
    print("Please run Sección 8 (training) first to generate checkpoints.")
    loaded_models = {}
else:
    checkpoint_files = sorted(checkpoints_dir.glob('fold_*.pt'))
    
    if len(checkpoint_files) == 0:
        print(f"\n⚠️  No checkpoint files found in {checkpoints_dir}")
        print("Expected files: fold_*.pt")
        print("Please run Sección 8 (training) first.")
        loaded_models = {}
    else:
        print(f"\nFound {len(checkpoint_files)} checkpoint(s). Loading...\n")
        
        loaded_models = {}
        reload_results = []
        
        for checkpoint_path in checkpoint_files:
            try:
                # Load checkpoint
                checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
                fold_name = checkpoint['fold_name']
                emb_dim = checkpoint['emb_dim']
                test_kinase = checkpoint['test_kinase']
                
                # Reconstruct model
                model_reloaded = SimpleEGNN(
                    embedding_dim=emb_dim,
                    node_dim=128,
                    num_layers=3,
                    hidden_dim=128
                ).to(DEVICE)
                
                # Load weights
                model_reloaded.load_state_dict(checkpoint['model_state_dict'])
                model_reloaded.eval()
                
                # Store in dictionary
                loaded_models[fold_name] = {
                    'model': model_reloaded,
                    'test_kinase': test_kinase,
                    'train_kinases': checkpoint['train_kinases'],
                    'test_metrics': checkpoint['test_metrics'],
                    'train_losses': checkpoint['train_losses'],
                    'val_losses': checkpoint['val_losses'],
                    'best_val_loss': checkpoint['best_val_loss'],
                    'emb_dim': emb_dim,
                }
                
                # Summary for table
                tm = checkpoint['test_metrics']
                reload_results.append({
                    'fold': fold_name,
                    'test_kinase': test_kinase,
                    'rmsd': tm['rmsd'],
                    'mae': tm['mae'],
                    'dist_err': tm['dist_error'],
                    'recovery': tm['recovery'],
                    'best_val_loss': checkpoint['best_val_loss'],
                })
                
                print(f"✓ {fold_name} ({test_kinase}): loaded successfully")
                
            except Exception as e:
                print(f"⚠️  Error loading {checkpoint_path.name}: {e}")
        
        # Display summary table
        if reload_results:
            print(f"\n{'='*80}")
            print("RELOADED MODELS SUMMARY")
            print(f"{'='*80}\n")
            reload_df = pd.DataFrame(reload_results)
            print(reload_df.to_string(index=False))
            print(f"\n✓ {len(loaded_models)} models successfully reloaded")
            print(f"✓ Ready for use in Sección 10 (equivariance test) or further analysis")
        else:
            print("\n⚠️  No models were successfully loaded.")
            loaded_models = {}

## 12. Batch Inference y Visualización con Modelos Reloaded

In [ ]:
"""
BATCH INFERENCE UTILITIES
Run predictions on all folds using reloaded models (or in-memory if available).
"""

def batch_inference_on_fold(model, test_pairs, fold_name, kinase_name, max_samples=None, device=DEVICE):
    """
    Run inference on test pairs for a fold and return predictions + metrics.
    
    Args:
        model: trained SimpleEGNN model
        test_pairs: list of pair dicts
        fold_name: str, for logging
        kinase_name: str, for logging
        max_samples: int or None (use all)
        device: torch device
    
    Returns:
        dict with predictions, metrics, etc.
    """
    if max_samples is None:
        max_samples = len(test_pairs)
    
    test_pairs_subset = test_pairs[:max_samples]
    
    predictions = []
    true_coords_list = []
    rmsd_list = []
    mae_list = []
    
    test_dataset = ConformationPairDataset(test_pairs_subset)
    
    with torch.no_grad():
        for i, pair in enumerate(test_pairs_subset):
            sample = test_dataset[i]
            
            inactive_coords = sample['inactive_coords'].numpy()
            inactive_embed = sample['inactive_embed'].numpy()
            inactive_adj = sample['inactive_adj'].numpy()
            true_coords = sample['active_coords'].numpy()
            valid_mask = sample['valid_mask'].numpy() > 0.5
            
            # Forward pass
            inp_embed = torch.from_numpy(inactive_embed).unsqueeze(0).to(device).float()
            inp_coords = torch.from_numpy(inactive_coords).unsqueeze(0).to(device).float()
            inp_adj = torch.from_numpy(inactive_adj).unsqueeze(0).to(device).float()
            
            pred_coords = model(inp_embed, inp_coords, inp_adj).squeeze(0).cpu().numpy()
            
            # Extract valid atoms
            pred_valid = pred_coords[valid_mask]
            true_valid = true_coords[valid_mask]
            
            # Metrics
            diff = pred_valid - true_valid
            rmsd_val = np.sqrt(np.mean((diff ** 2).sum(axis=1)))
            mae_val = np.mean(np.abs(diff))
            
            predictions.append({
                'pair_idx': i,
                'predicted_coords': pred_valid,
                'true_coords': true_valid,
                'rmsd': rmsd_val,
                'mae': mae_val,
            })
            
            rmsd_list.append(rmsd_val)
            mae_list.append(mae_val)
    
    return {
        'fold_name': fold_name,
        'kinase_name': kinase_name,
        'n_samples': len(test_pairs_subset),
        'predictions': predictions,
        'mean_rmsd': np.mean(rmsd_list) if rmsd_list else 0.0,
        'mean_mae': np.mean(mae_list) if mae_list else 0.0,
        'rmsd_list': rmsd_list,
        'mae_list': mae_list,
    }


# Example: Run batch inference on first loaded model
if 'loaded_models' in locals() and len(loaded_models) > 0:
    print("\n" + "=" * 80)
    print("BATCH INFERENCE EXAMPLE (first fold)")
    print("=" * 80 + "\n")
    
    fold_name = sorted(loaded_models.keys())[0]
    model_info = loaded_models[fold_name]
    model = model_info['model']
    test_kinase = model_info['test_kinase']
    
    # Note: test_pairs are not persisted in checkpoints (too large)
    # so we can only do inference on newly constructed pairs from dataset
    # This is a limitation but models can still be evaluated on any new data
    
    print(f"✓ Inference on fold: {fold_name} (test kinase: {test_kinase})")
    print(f"✓ Model embedding_dim: {model_info['emb_dim']}")
    print(f"✓ Test metrics from training: RMSD={model_info['test_metrics']['rmsd']:.4f} Å, "
          f"MAE={model_info['test_metrics']['mae']:.4f} Å")
    
    print("\n✓ Reloaded models are ready for:")
    print("  - Further evaluation on new data")
    print("  - Structural predictions")
    print("  - Equivariance testing (Sección 10)")
    print("  - Downstream analysis and visualization")
    
else:
    print("\n⚠️  No loaded models available yet.")
    print("Run Sección 11 (reload checkpoints) first.")